FEATURE ENGINEERING

We are doing to look at the following

Prepare the Forecasting Dataset
Create Calendar Features
Create Historical Demand Lag Features
Create Rolling Demand Features
Prepare Planned-Pour Information
Prepare Weather Information
Prepare Site and Cement-Type Information
Validate Feature Availability and Target Leakage
Final Feature Set and Save Modeling Dataset

In [1]:
import pandas as pd

In [2]:
# Load the cleaned
forecast_data = pd.read_csv(r"C:\Users\hp\Desktop\Amdari Internship\Projects\Cement Forecasting\data\cleaned_dataset.csv", parse_dates=["date"])
forecast_data.head()

,date,site_id,cement_type,planned_pour_tonnes,consumed_tonnes,opening_inventory_tonnes,deliveries_tonnes,closing_inventory_tonnes,rain_mm,avg_temp_c,silo_capacity,region,behavior,opening_capacity_violation,closing_capacity_violation
0,2022-01-01,SITE_001,CEM_II,43.18,34.54,52.56,45.83,63.85,3.40,-3.10,448,North,aggressive,False,False
1,2022-01-02,SITE_001,CEM_I,45.26,45.26,63.85,19.97,38.56,3.23,14.28,448,North,aggressive,False,False
2,2022-01-03,SITE_001,CEM_III,38.69,38.69,38.56,47.19,47.06,2.64,6.40,448,North,aggressive,False,False
3,2022-01-04,SITE_001,CEM_I,33.16,33.16,47.06,18.74,32.64,8.25,14.23,448,North,aggressive,False,False
4,2022-01-05,SITE_001,CEM_III,56.88,47.04,32.64,14.40,0.00,2.69,8.97,448,North,aggressive,False,False


In [3]:
forecast_data.shape

(32880, 15)

In [4]:
# Sorting the data chronologically by date
forecast_data = forecast_data.sort_values(
    ["site_id", "cement_type", "date"]
).reset_index(drop=True)

In [5]:
# Let Create Calendar Features

forecast_data["year"] = forecast_data["date"].dt.year
forecast_data["month"] = forecast_data["date"].dt.month
forecast_data["week_of_year"] = (
    forecast_data["date"]
    .dt.isocalendar()
    .week
    .astype(int)
)
forecast_data["day_of_week"] = (
    forecast_data["date"].dt.dayofweek
)

 #### Create Historical Demand Lag Features

Historical cement consumption may provide useful information about future site demand.


In [6]:
forecast_data["demand_lag_1d"] = (
    forecast_data.groupby("site_id")["consumed_tonnes"]
    .shift(1)
)

forecast_data["demand_lag_7d"] = (
    forecast_data.groupby("site_id")["consumed_tonnes"]
    .shift(7)
)

forecast_data["demand_lag_14d"] = (
    forecast_data.groupby("site_id")["consumed_tonnes"]
    .shift(14)
)

forecast_data["demand_lag_28d"] = (
    forecast_data.groupby("site_id")["consumed_tonnes"]
    .shift(28)
)

#### Rolling Demand Features

In [7]:
forecast_data["rolling_mean_7d"] = (
    forecast_data.groupby("site_id")["consumed_tonnes"]
    .transform(
        lambda x: x.shift(1).rolling(7, min_periods=7).mean()
    )
)

forecast_data["rolling_mean_14d"] = (
    forecast_data.groupby("site_id")["consumed_tonnes"]
    .transform(
        lambda x: x.shift(1).rolling(14, min_periods=14).mean()
    )
)

forecast_data["rolling_mean_28d"] = (
    forecast_data.groupby("site_id")["consumed_tonnes"]
    .transform(
        lambda x: x.shift(1).rolling(28, min_periods=28).mean()
    )
)

forecast_data["rolling_std_7d"] = (
    forecast_data.groupby("site_id")["consumed_tonnes"]
    .transform(
        lambda x: x.shift(1).rolling(7, min_periods=7).std()
    )
)

In [9]:
forecast_data["rolling_mean_7d"] = (
    forecast_data.groupby("site_id")["consumed_tonnes"]
    .transform(lambda x: x.shift(1).rolling(window=7).mean())
)

forecast_data["rolling_mean_28d"] = (
    forecast_data.groupby("site_id")["consumed_tonnes"]
    .transform(lambda x: x.shift(1).rolling(window=28).mean())
)

#### Defining the forecast features

In [15]:
target_column = "consumed_tonnes"

identifier_columns = ["date"]

feature_columns = [
    "site_id",
    "cement_type",
    "planned_pour_tonnes",
    "rain_mm",
    "avg_temp_c",
    "silo_capacity",
    "region",
    "behavior",
    "year",
    "month",
    "week_of_year",
    "day_of_week",
    "demand_lag_1d",
    "demand_lag_7d",
    "demand_lag_14d",
    "demand_lag_28d",
    "rolling_mean_7d",
    "rolling_mean_14d",
    "rolling_mean_28d",
    "rolling_std_7d"
]

inventory_engine_columns = [
    "opening_inventory_tonnes",
    "deliveries_tonnes",
    "closing_inventory_tonnes",
    "opening_capacity_violation",
    "closing_capacity_violation"
]

- These are the forecast features, I am excluding the inventory variables. We are only preventing them from entering the demand-forecasting model

##### Saving the final modelling dataset

In [17]:
modeling_data = (
    forecast_data[
        identifier_columns + feature_columns + [target_column]
    ]
    .dropna()
    .sort_values(["date", "site_id"])
    .reset_index(drop=True)
)

In [20]:
# let's save the model dataset
modeling_data.to_csv(
    "../data/processed/cement_forecasting_model_data.csv",
    index=False
)